# RAG Pipeline — BCTC ACBS Project

Pipeline: chunks.jsonl → Vietnamese embedding (E5 prefix) → ChromaDB → Hybrid retrieval (BM25 + Dense + Ticker boost) → Gemini answer.

**2 tests cuối:**
1. Stress test 10 queries với scoring top-1 đúng nguồn
2. RAG full pipeline: query → answer + citation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q sentence-transformers chromadb rank_bm25 google-genai
!nvidia-smi | head -10

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 128.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 162.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 141.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 8.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are insta

In [2]:
import json
from collections import Counter

CHUNKS_PATH = "/content/drive/MyDrive/chunks.jsonl"
chunks = []
with open(CHUNKS_PATH, encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"Total chunks: {len(chunks)}")
print(f"  Table: {sum(1 for c in chunks if c['block_type']=='table')}")
print(f"  Text : {sum(1 for c in chunks if c['block_type']=='text')}")
print(f"Source files: {len(set(c['source'] for c in chunks))}")

Total chunks: 6487
  Table: 2858
  Text : 3629
Source files: 40


In [3]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'AITeamVN/Vietnamese_Embedding'
print(f"Loading {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME, trust_remote_code=True, device='cuda')
print(f"Embedding dim: {model.get_sentence_embedding_dimension()} | Max seq: {model.max_seq_length}\n")

# E5/Halong family yêu cầu prefix 'passage:' cho documents, 'query:' cho queries
passages = [f"passage: {c['text']}" for c in chunks]
print(f"Embedding {len(passages)} chunks...")
embeddings = model.encode(
    passages, batch_size=32, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
)
print(f"\nEmbeddings shape: {embeddings.shape}")

Loading AITeamVN/Vietnamese_Embedding...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Embedding dim: 1024 | Max seq: 8192

Embedding 6487 chunks...


/tmp/ipykernel_1284/1678411117.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dim: {model.get_sentence_embedding_dimension()} | Max seq: {model.max_seq_length}\n")


Batches:   0%|          | 0/203 [00:00<?, ?it/s]


Embeddings shape: (6487, 1024)


In [4]:
import chromadb, shutil, os

LOCAL_DB = "/content/chromadb"
DRIVE_DB = "/content/drive/MyDrive/chromadb"
COLL_NAME = "bctc_acbs"

# Tạo trên local disk trước (chromadb ghi SQLite, Drive FUSE chậm + có thể lock)
if os.path.exists(LOCAL_DB):
    shutil.rmtree(LOCAL_DB)
os.makedirs(LOCAL_DB)

client = chromadb.PersistentClient(path=LOCAL_DB)
collection = client.get_or_create_collection(
    name=COLL_NAME,
    metadata={"hnsw:space": "cosine"},
)

def flatten_meta(c):
    m = {"source": c['source'], "page": c['page'], "block_type": c['block_type']}
    if 'block_id' in c: m['block_id'] = c['block_id']
    if 'block_ids' in c: m['block_ids'] = ",".join(map(str, c['block_ids']))
    if 'bbox' in c and c['bbox']: m['bbox'] = ",".join(map(str, c['bbox']))
    if 'split_idx' in c: m['split_idx'] = c['split_idx']
    if 'table_part' in c: m['table_part'] = c['table_part']
    return m

ids = [f"{c['source']}_p{c['page']}_{i}" for i, c in enumerate(chunks)]
docs = [c['text'] for c in chunks]
metas = [flatten_meta(c) for c in chunks]

BATCH = 1000
for i in range(0, len(ids), BATCH):
    end = min(i + BATCH, len(ids))
    collection.upsert(
        ids=ids[i:end],
        embeddings=embeddings[i:end].tolist(),
        documents=docs[i:end],
        metadatas=metas[i:end],
    )
    print(f"  Upserted {end}/{len(ids)}")

print(f"\nLocal collection count: {collection.count()}")

# Copy DB sang Drive để dùng lại sau (cho RAG agent ngày 6)
print("\nCopying ChromaDB → Drive (~30s)...")
if os.path.exists(DRIVE_DB):
    shutil.rmtree(DRIVE_DB)
shutil.copytree(LOCAL_DB, DRIVE_DB)
print(f"Saved to: {DRIVE_DB}")
print(f"Size: {sum(f.stat().st_size for f in os.scandir(DRIVE_DB) if f.is_file()) / 1024 / 1024:.1f} MB")

  Upserted 1000/6487
  Upserted 2000/6487
  Upserted 3000/6487
  Upserted 4000/6487
  Upserted 5000/6487
  Upserted 6000/6487
  Upserted 6487/6487

Local collection count: 6487

Copying ChromaDB → Drive (~30s)...
Saved to: /content/drive/MyDrive/chromadb
Size: 52.4 MB


In [5]:
import re
from rank_bm25 import BM25Okapi
from collections import defaultdict

VN_CHARS = 'a-z0-9àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ'
token_re = re.compile(f'[{VN_CHARS}]+')
html_re = re.compile(r'<[^>]+>')

def tokenize(text):
    text = html_re.sub(' ', text).lower()
    return token_re.findall(text)

# BM25 index — thêm source vào corpus để bắt ticker
corpus_tokens = [tokenize(c['text'] + ' ' + c['source']) for c in chunks]
bm25 = BM25Okapi(corpus_tokens)
print(f"BM25 indexed: {len(corpus_tokens)} chunks")

# Map full name → ticker để query "Vinamilk" match được "VNM"
TICKER_NAMES = {
    'HPG': ['hòa phát', 'hoa phat'], 'VCB': ['vietcombank'],
    'VNM': ['vinamilk'], 'VHC': ['vĩnh hoàn', 'vinh hoan'],
    'VHM': ['vinhomes'], 'VIC': ['vingroup'], 'VRE': ['vincom retail'],
    'FPT': ['fpt'], 'BID': ['bidv'], 'CTG': ['vietinbank'],
    'TCB': ['techcombank'], 'MBB': ['mb bank', 'mbbank'],
    'STB': ['sacombank'], 'VPB': ['vpbank'], 'HDB': ['hdbank'],
    'VIB': ['vib'], 'TPB': ['tpbank'], 'LPB': ['lpbank'],
    'SSB': ['seabank'], 'ACB': ['acb'], 'BVH': ['bảo việt', 'bao viet'],
    'GAS': ['pv gas'], 'PLX': ['petrolimex'], 'POW': ['pv power'],
    'GVR': ['cao su'], 'DGC': ['đức giang'], 'MSN': ['masan'],
    'NVL': ['novaland'], 'PDR': ['phát đạt'], 'KDH': ['khang điền'],
    'SAB': ['sabeco'], 'BCM': ['becamex'],
}

def expand_query(q):
    q_low = q.lower()
    extra = [t for t, names in TICKER_NAMES.items()
             if any(n in q_low for n in names) and t.lower() not in q_low]
    return q + (' ' + ' '.join(extra) if extra else '')

def extract_ticker(query):
    q_low = query.lower()
    for ticker in TICKER_NAMES.keys():
        if re.search(rf'\b{ticker.lower()}\b', q_low):
            return ticker
    for ticker, names in TICKER_NAMES.items():
        if any(n in q_low for n in names):
            return ticker
    return None

def hybrid_search(query, top_k=3, k_rrf=60, candidates=20, source_boost=2.0):
    """BM25 + Dense + RRF fusion + ticker source boost"""
    expanded = expand_query(query)
    ticker = extract_ticker(query)

    bm25_scores = bm25.get_scores(tokenize(expanded))
    bm25_top = sorted(range(len(bm25_scores)), key=lambda i: -bm25_scores[i])[:candidates]

    q_emb = model.encode([f"query: {expanded}"], normalize_embeddings=True, convert_to_numpy=True)
    dense_res = collection.query(query_embeddings=q_emb.tolist(), n_results=candidates)
    dense_top = [int(id_.rsplit('_', 1)[-1]) for id_ in dense_res['ids'][0]]

    rrf = defaultdict(float)
    def add(rank, idx):
        s = 1.0 / (k_rrf + rank)
        if ticker and ticker.upper() in chunks[idx]['source'].upper():
            s *= source_boost
        rrf[idx] += s

    for rank, idx in enumerate(bm25_top): add(rank, idx)
    for rank, idx in enumerate(dense_top): add(rank, idx)

    return sorted(rrf.items(), key=lambda x: -x[1])[:top_k]

BM25 indexed: 6487 chunks


## TEST 1 — Stress test 10 queries với scoring

In [6]:
QUERIES = [
    "Lợi nhuận trước thuế của Techcombank quý 1 năm 2026",
    "ROE của Vietcombank Q1/2026 là bao nhiêu",
    "Doanh thu mảng vận tải biển của HAH năm 2026",          # edge: HAH không có file riêng
    "Vốn điều lệ của Vingroup VIC",
    "Thuế quan Hoa Kỳ ảnh hưởng Vĩnh Hoàn cá tra ra sao",
    "ACBS định giá mục tiêu cổ phiếu DPM bao nhiêu",
    "FTSE Russell nâng hạng thị trường Việt Nam khi nào",
    "Tổng tài sản của ngân hàng MB Bank cuối Q1/2026",
    "So sánh nợ vay ngắn hạn STB và HDB Q1/2026",
    "ACBS khuyến nghị mua hay bán cổ phiếu Masan MSN",
]

EXPECTED = {
    1: "TCB", 2: "VCB", 3: "BSC|VCSC", 4: "VIC",
    5: "VHC", 6: "ACBS_DPM", 7: "BSC|VCSC",
    8: "MBB", 9: "STB|HDB", 10: "ACBS_MSN",
}

print("STRESS TEST: 10 câu, top-3 mỗi câu\n")
correct = 0
for q_idx, q in enumerate(QUERIES, 1):
    results = hybrid_search(q, top_k=3)
    print("=" * 78)
    print(f"Q{q_idx}: {q}")
    print(f"     ticker={extract_ticker(q)}")
    print("-" * 78)
    expected_pat = EXPECTED[q_idx]
    top1_source = chunks[results[0][0]]['source']
    is_correct = any(p.upper() in top1_source.upper() for p in expected_pat.split('|'))
    correct += int(is_correct)
    for rank, (idx, score) in enumerate(results, 1):
        c = chunks[idx]
        snippet = c['text'][:180].replace('\n', ' ')
        mark = " ✓" if rank == 1 and is_correct else ""
        print(f"  [{rank}] rrf={score:.4f} | {c['source']} p{c['page']} ({c['block_type']}){mark}")
        print(f"      {snippet}{'...' if len(c['text'])>180 else ''}")
    print()

print("=" * 78)
print(f"STRESS TEST SCORE: {correct}/10 top-1 đúng nguồn")

STRESS TEST: 10 câu, top-3 mỗi câu

Q1: Lợi nhuận trước thuế của Techcombank quý 1 năm 2026
     ticker=TCB
------------------------------------------------------------------------------
  [1] rrf=0.0317 | TCB p30 (table) ✓
      <table border=1><tr><td></td><td>Tỷ lệ phần trăm của lợi nhuận sau thuế</td><td>Mức tối đa</td></tr><tr><td>Quỹ dự trữ bỏ sung vốn điều lệ</td><td>10% lợi nhuận sau thuế sau khi ch...
  [2] rrf=0.0306 | CTG p49 (text)
      ## 24. GIẢI TRÌNH BIẾN ĐỘNG LỘI NHUẬN  Lợi nhuận sau thuế TNDN hợp nhất Quý I năm 2026 của VietinBank tăng 3.461 tỷ đồng (tương đương mức tăng 62,93%) so với cùng kỳ năm 2025 nguyê...
  [3] rrf=0.0296 | BCM p2 (table)
      <table border=1><tr><td>STT</td><td>Khoản mục</td><td>Quý 1 năm 2026 (đồng)</td><td>Quý 1 năm 2025 (đồng)</td><td>Tỷ lệ (%)</td></tr><tr><td>1</td><td>Lợi nhuận sau thuế TNDN - BCT...

Q2: ROE của Vietcombank Q1/2026 là bao nhiêu
     ticker=VCB
----------------------------------------------------------------------------

## TEST 2 — RAG full pipeline (Gemini answer + citation)

In [7]:
from google.colab import userdata
from google import genai
import os

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
client_llm = genai.Client()

def strip_html_for_llm(text):
    """Convert HTML table to readable plain text for LLM"""
    text = re.sub(r'</tr>\s*<tr[^>]*>', '\n', text)
    text = re.sub(r'</td>\s*<td[^>]*>', ' | ', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def rag_answer(query, top_k=5, model_name="gemini-2.5-flash"):
    results = hybrid_search(query, top_k=top_k)

    context_parts, sources_info = [], []
    for i, (idx, _) in enumerate(results, 1):
        c = chunks[idx]
        readable = strip_html_for_llm(c['text'])
        context_parts.append(
            f"[Chunk {i}] Nguồn: {c['source']} (trang {c['page']}, loại: {c['block_type']})\n{readable}"
        )
        sources_info.append(f"[Chunk {i}] {c['source']} p{c['page']} ({c['block_type']})")

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""Bạn là chuyên gia phân tích tài chính của ACBS. Trả lời câu hỏi dựa CHỈ TRÊN context được cung cấp.

Yêu cầu nghiêm ngặt:
1. Trả lời ngắn gọn, chính xác bằng tiếng Việt.
2. MỌI thông tin và số liệu phải kèm citation dạng [Chunk N].
3. Nếu context không đủ để trả lời, nói rõ: "Không tìm thấy thông tin trong tài liệu được cung cấp."
4. KHÔNG bịa số liệu, KHÔNG suy đoán ngoài context.
5. Số tiền giữ nguyên format gốc (dấu chấm phân cách hàng nghìn).

=== CONTEXT ===
{context}

=== CÂU HỎI ===
{query}

=== TRẢ LỜI ==="""

    response = client_llm.models.generate_content(model=model_name, contents=prompt)

    print(f"{query}\n")
    print(f"{response.text}\n")
    print("Sources retrieved:")
    for s in sources_info:
        print(f"  {s}")
    return response.text

# Test với câu specific (TCB chi tiết)
rag_answer("Dự phòng rủi ro trái phiếu doanh nghiệp chưa niêm yết ngày 31/3/2026 của ngân hàng TMCP Kỹ thương Việt Nam Techcombank là bao nhiêu")

Dự phòng rủi ro trái phiếu doanh nghiệp chưa niêm yết ngày 31/3/2026 của ngân hàng TMCP Kỹ thương Việt Nam Techcombank là bao nhiêu

Dự phòng rủi ro trái phiếu doanh nghiệp chưa niêm yết của Ngân hàng TMCP Kỹ thương Việt Nam Techcombank ngày 31/3/2026 là 415.382 triệu đồng [Chunk 1].

Sources retrieved:
  [Chunk 1] TCB p47 (table)
  [Chunk 2] TCB p16 (text)
  [Chunk 3] TCB p23 (text)
  [Chunk 4] TCB p15 (text)
  [Chunk 5] TCB p55 (table)


'Dự phòng rủi ro trái phiếu doanh nghiệp chưa niêm yết của Ngân hàng TMCP Kỹ thương Việt Nam Techcombank ngày 31/3/2026 là 415.382 triệu đồng [Chunk 1].'

In [11]:
rag_answer("Tiền thu từ đi vay của BCM là bao nhiêu ")
rag_answer("Tiền thu từ đi vay (năm trước) của BCM là bao nhiêu")

Tiền thu từ đi vay của BCM là bao nhiêu 

Tiền thu từ đi vay của BCM là 4.853.789.760.059 [Chunk 2].

Sources retrieved:
  [Chunk 1] BCM p28 (text)
  [Chunk 2] BCM p11 (table)
  [Chunk 3] BCM p11 (table)
  [Chunk 4] BCM p27 (text)
  [Chunk 5] BCM p29 (text)
Tiền thu từ đi vay (năm trước) của BCM là bao nhiêu

Tiền thu từ đi vay của BCM (năm trước) là 850.247.940.423 [Chunk 1].

Sources retrieved:
  [Chunk 1] BCM p11 (table)
  [Chunk 2] BCM p28 (text)
  [Chunk 3] BCM p27 (text)
  [Chunk 4] BCM p34 (table)
  [Chunk 5] PDR p9 (table)


'Tiền thu từ đi vay của BCM (năm trước) là 850.247.940.423 [Chunk 1].'

In [13]:
rag_answer("Lợi thế thương mại của VHM như thế nào")


Lợi thế thương mại của VHM như thế nào

Lợi thế thương mại của VHM được ghi nhận theo phương pháp giả mua và ban đầu xác định theo giá gốc, là phần chênh lệch giữa giá phí hợp nhất kinh doanh so với phần sở hữu của bên mua trong giá trị hợp lý của tài sản, nợ phải trả có thể xác định được và các khoản nợ tiềm tàng đã ghi nhận [Chunk 1].

Sau ghi nhận ban đầu, lợi thế thương mại được xác định giá trị bằng nguyên giá trừ giá trị phân bổ lũy kế. Lợi thế thương mại được phân bổ theo phương pháp đường thẳng trong thời gian hữu ích ước tính là mười (10) năm. Định kỳ, công ty mẹ phải đánh giá tổn thất lợi thế thương mại [Chunk 1].

Các số liệu về lợi thế thương mại của VHM như sau:
*   **Nguyên giá (Original cost):**
    *   Vào ngày 31 tháng 12 năm 2025: 1.466.560 [Chunk 2]
    *   Vào ngày 31 tháng 3 năm 2025: 1.466.560 [Chunk 2]
*   **Phân bổ lũy kế (Accumulated amortization):**
    *   Vào ngày 31 tháng 12 năm 2025: 1.172.255 [Chunk 2]
    *   Phân bổ trong kỳ (Amortization during the per

'Lợi thế thương mại của VHM được ghi nhận theo phương pháp giả mua và ban đầu xác định theo giá gốc, là phần chênh lệch giữa giá phí hợp nhất kinh doanh so với phần sở hữu của bên mua trong giá trị hợp lý của tài sản, nợ phải trả có thể xác định được và các khoản nợ tiềm tàng đã ghi nhận [Chunk 1].\n\nSau ghi nhận ban đầu, lợi thế thương mại được xác định giá trị bằng nguyên giá trừ giá trị phân bổ lũy kế. Lợi thế thương mại được phân bổ theo phương pháp đường thẳng trong thời gian hữu ích ước tính là mười (10) năm. Định kỳ, công ty mẹ phải đánh giá tổn thất lợi thế thương mại [Chunk 1].\n\nCác số liệu về lợi thế thương mại của VHM như sau:\n*   **Nguyên giá (Original cost):**\n    *   Vào ngày 31 tháng 12 năm 2025: 1.466.560 [Chunk 2]\n    *   Vào ngày 31 tháng 3 năm 2025: 1.466.560 [Chunk 2]\n*   **Phân bổ lũy kế (Accumulated amortization):**\n    *   Vào ngày 31 tháng 12 năm 2025: 1.172.255 [Chunk 2]\n    *   Phân bổ trong kỳ (Amortization during the period): 37.724 [Chunk 2]\n    *

In [16]:
rag_answer("BSC nói gì về thị trường cơ sở ngày 5 tháng 5 năm 2026")

BSC nói gì về thị trường cơ sở ngày 5 tháng 5 năm 2026

Vào ngày 5 tháng 5 năm 2026, về thị trường cơ sở, BSC báo cáo những điểm chính sau:
*   VN-Index tăng gần 21 điểm (+20.79 điểm), đóng cửa tại 1874.85 điểm, với sự đóng góp chủ yếu từ nhóm cổ phiếu Vin (VIC (+12.71), VHM (+8.51), VPL (+1.10)) [Chunk 4].
*   HNX-Index giảm 2.62 điểm, đóng cửa tại 247.42 điểm [Chunk 4].
*   Độ rộng thị trường nghiêng về phía tiêu cực với 12/18 ngành giảm điểm; ngành Hóa chất giảm mạnh nhất, theo sau là Bảo hiểm, Hàng cá nhân & gia dụng [Chunk 4].
*   Ngược lại, ngành Dầu khí tăng mạnh nhất, theo sau là Bất động sản [Chunk 4].
*   Giá trị khớp lệnh của VN-Index đạt 18,965 tỷ đồng, tăng 11.63% so với phiên trước. Tổng giá trị giao dịch đạt 22,395 tỷ đồng [Chunk 4].
*   Thị trường có 97 mã tăng, 57 mã tham chiếu, 213 mã giảm [Chunk 4].
*   Về giao dịch của khối ngoại: bán ròng -995.75 tỷ đồng trên sàn HOSE, chủ yếu ở ACB (-426.42 tỷ), HPG (-284.55 tỷ), FPT (-133.89 tỷ), VCB (-87.44 tỷ), KDH (-71.65 tỷ).

'Vào ngày 5 tháng 5 năm 2026, về thị trường cơ sở, BSC báo cáo những điểm chính sau:\n*   VN-Index tăng gần 21 điểm (+20.79 điểm), đóng cửa tại 1874.85 điểm, với sự đóng góp chủ yếu từ nhóm cổ phiếu Vin (VIC (+12.71), VHM (+8.51), VPL (+1.10)) [Chunk 4].\n*   HNX-Index giảm 2.62 điểm, đóng cửa tại 247.42 điểm [Chunk 4].\n*   Độ rộng thị trường nghiêng về phía tiêu cực với 12/18 ngành giảm điểm; ngành Hóa chất giảm mạnh nhất, theo sau là Bảo hiểm, Hàng cá nhân & gia dụng [Chunk 4].\n*   Ngược lại, ngành Dầu khí tăng mạnh nhất, theo sau là Bất động sản [Chunk 4].\n*   Giá trị khớp lệnh của VN-Index đạt 18,965 tỷ đồng, tăng 11.63% so với phiên trước. Tổng giá trị giao dịch đạt 22,395 tỷ đồng [Chunk 4].\n*   Thị trường có 97 mã tăng, 57 mã tham chiếu, 213 mã giảm [Chunk 4].\n*   Về giao dịch của khối ngoại: bán ròng -995.75 tỷ đồng trên sàn HOSE, chủ yếu ở ACB (-426.42 tỷ), HPG (-284.55 tỷ), FPT (-133.89 tỷ), VCB (-87.44 tỷ), KDH (-71.65 tỷ). Khối ngoại mua ròng trên sàn HNX với giá trị 14

In [17]:
rag_answer("Ngày 4/5/2026, VCSC cho báo cáo về dữ liệu thị trường như thế nào")

Ngày 4/5/2026, VCSC cho báo cáo về dữ liệu thị trường như thế nào

Ngày 4/5/2026, VCSC đã báo cáo dữ liệu thị trường như sau:
*   VN-Index chốt phiên gần như đi ngang (-0,04 điểm; 0,0%), đóng cửa ở mức 1.854,1 điểm [Chunk 4].
*   Diễn biến phân hóa diễn ra ở hầu hết các nhóm ngành [Chunk 4].
*   Toàn sàn ghi nhận 169 mã tăng, 148 mã giảm và 60 mã đóng cửa tham chiếu [Chunk 4].
*   Thanh khoản tăng nhẹ so với phiên trước, đạt 21,2 nghìn tỷ đồng [Chunk 4].
*   Trong nhóm Bất động sản, VRE (+4,3%) và KBC (+1,6%) tăng mạnh, trong khi NVL (-6,8%), VHM (-2,7%) và VIC (-0,9%) giảm điểm [Chunk 4].
*   Trong nhóm Ngân hàng, VCB (+1,5%), BID (+1,9%), CTG (+1,1%) và VPB (+1,7%) ghi nhận mức tăng tốt. Ngược lại, STB (-1,9%), ACB (-1,7%) và TCB (-0,9%) suy giảm [Chunk 4].
*   Nhóm cổ phiếu Dầu khí diễn biến tích cực khi BSR (+7,0%), PLX (+4,2%) và PVD (+2,4%) đều tăng điểm [Chunk 4].
*   Các mã tăng đáng chú ý ở các lĩnh vực khác bao gồm GVR (+7,0%), GAS (+1,7%), POW (+5,1%), MWG (+1,1%) và PNJ (+1

'Ngày 4/5/2026, VCSC đã báo cáo dữ liệu thị trường như sau:\n*   VN-Index chốt phiên gần như đi ngang (-0,04 điểm; 0,0%), đóng cửa ở mức 1.854,1 điểm [Chunk 4].\n*   Diễn biến phân hóa diễn ra ở hầu hết các nhóm ngành [Chunk 4].\n*   Toàn sàn ghi nhận 169 mã tăng, 148 mã giảm và 60 mã đóng cửa tham chiếu [Chunk 4].\n*   Thanh khoản tăng nhẹ so với phiên trước, đạt 21,2 nghìn tỷ đồng [Chunk 4].\n*   Trong nhóm Bất động sản, VRE (+4,3%) và KBC (+1,6%) tăng mạnh, trong khi NVL (-6,8%), VHM (-2,7%) và VIC (-0,9%) giảm điểm [Chunk 4].\n*   Trong nhóm Ngân hàng, VCB (+1,5%), BID (+1,9%), CTG (+1,1%) và VPB (+1,7%) ghi nhận mức tăng tốt. Ngược lại, STB (-1,9%), ACB (-1,7%) và TCB (-0,9%) suy giảm [Chunk 4].\n*   Nhóm cổ phiếu Dầu khí diễn biến tích cực khi BSR (+7,0%), PLX (+4,2%) và PVD (+2,4%) đều tăng điểm [Chunk 4].\n*   Các mã tăng đáng chú ý ở các lĩnh vực khác bao gồm GVR (+7,0%), GAS (+1,7%), POW (+5,1%), MWG (+1,1%) và PNJ (+1,5%) [Chunk 4].\n*   Các mã giảm nổi bật gồm FPT (-2,4%), 

In [18]:
rag_answer("Ngày 4/5/2026, VCSC nói gì về Áp lực lạm phát ảnh hưởng ngành sản xuất Việt Nam ")

Ngày 4/5/2026, VCSC nói gì về Áp lực lạm phát ảnh hưởng ngành sản xuất Việt Nam 

Ngày 4/5/2026, VCSC nhận định về áp lực lạm phát ảnh hưởng ngành sản xuất Việt Nam như sau:
*   Chỉ số PMI ngành sản xuất Việt Nam của S&P Global trong tháng 4 đã giảm xuống 50,5 từ mức 51,2 trong tháng 3, đây là mức thấp nhất trong 7 tháng qua [Chunk 1].
*   Tốc độ tăng trưởng sản lượng đã chậm lại ở mức yếu nhất kể từ tháng 6/2025, phản ánh áp lực lạm phát gia tăng, tình trạng thắt chặt nguồn cung và sự bất ổn của thị trường liên quan đến xung đột tại Trung Đông [Chunk 1].
*   Chi phí đầu vào và giá đầu ra tăng với tốc độ nhanh nhất trong 15 năm [Chunk 1].
*   Mức giá cao hơn bắt đầu gây áp lực lên nhu cầu, khiến số lượng đơn đặt hàng mới giảm lần đầu tiên trong 8 tháng qua [Chunk 1].
*   Để ứng phó, các doanh nghiệp đã cắt giảm cả hoạt động mua hàng và tuyển dụng [Chunk 1].
*   Niềm tin kinh doanh cũng suy yếu xuống mức thấp nhất trong 7 tháng [Chunk 1].
*   Nhìn chung, áp lực chi phí có khả năng sẽ gâ

'Ngày 4/5/2026, VCSC nhận định về áp lực lạm phát ảnh hưởng ngành sản xuất Việt Nam như sau:\n*   Chỉ số PMI ngành sản xuất Việt Nam của S&P Global trong tháng 4 đã giảm xuống 50,5 từ mức 51,2 trong tháng 3, đây là mức thấp nhất trong 7 tháng qua [Chunk 1].\n*   Tốc độ tăng trưởng sản lượng đã chậm lại ở mức yếu nhất kể từ tháng 6/2025, phản ánh áp lực lạm phát gia tăng, tình trạng thắt chặt nguồn cung và sự bất ổn của thị trường liên quan đến xung đột tại Trung Đông [Chunk 1].\n*   Chi phí đầu vào và giá đầu ra tăng với tốc độ nhanh nhất trong 15 năm [Chunk 1].\n*   Mức giá cao hơn bắt đầu gây áp lực lên nhu cầu, khiến số lượng đơn đặt hàng mới giảm lần đầu tiên trong 8 tháng qua [Chunk 1].\n*   Để ứng phó, các doanh nghiệp đã cắt giảm cả hoạt động mua hàng và tuyển dụng [Chunk 1].\n*   Niềm tin kinh doanh cũng suy yếu xuống mức thấp nhất trong 7 tháng [Chunk 1].\n*   Nhìn chung, áp lực chi phí có khả năng sẽ gây áp lực lên sản xuất và số lượng đơn hàng mới, điều này có thể tác động t